# YOLO11 — Helmet + Electric-Vehicle detector

Notebook này train HAI bài toán riêng biệt (handoff R9):
- Helmet detection (with_helmet / no_helmet / occluded / unknown)
- Electric vehicle classification (electric / fuel_motorcycle)

KHÔNG ép model gộp nhiều lớp. Mỗi task có dataset + manifest riêng, train
riêng, export riêng. Owner B chạy tuần tự 2 cell train.

Mapping helmet 4 lớp: `with_helmet, no_helmet, occluded, unknown`.
Mapping xe điện 2 lớp: `electric, fuel_motorcycle`. Cần nguồn dữ liệu riêng —
chưa có trong dataset gốc (helmet_detect = 0 ảnh).


In [ ]:
import os
# Kaggle environment: pin Ultralytics, torch, fast_plate_ocr theo kaggle image.
# KHÔNG ép nâng cấp Torch local — pin đúng version đang dùng (YOLO11 cần ≥8.4).
os.environ.setdefault("YOLO_CONFIG_DIR", "/kaggle/working/yolo_cfg")
os.environ.setdefault("ULTRALYTICS_RUNS_DIR", "/kaggle/working/runs")
os.environ.setdefault("ULTRALYTICS_OFFLINE", "True")
os.environ.setdefault("ULTRALYTICS_TELEMETRY", "False")
# Tắt mọi plotting/caching Ultralytics
os.environ.setdefault("YOLO_VERBOSE", "False")
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")


In [ ]:
# Kaggle /kaggle/working là output dir. Tất cả artifact phải vào đây.
import os
from pathlib import Path
WORKDIR = Path(os.environ.get("WORKDIR", "/kaggle/working"))
WORKDIR.mkdir(parents=True, exist_ok=True)
print("workdir:", WORKDIR)


In [ ]:
def preflight(audit_dir: str) -> dict:
    """Kiểm tra manifest + hash + group-disjoint trước khi train.

    Trả về dict với 'ok' bool + 'pending_data' nếu thiếu.
    Raise nếu hash khác (file đã bị sửa sau export).
    """
    import hashlib
    import os
    import json

    manifest_path = os.path.join(audit_dir, "manifest.json")
    if not os.path.isfile(manifest_path):
        return {"ok": False, "pending_data": True,
                "reason": f"manifest.json không có tại {audit_dir}"}
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    # Manifest.sha256 đi kèm file đã verify ở R8; ở đây chỉ in lại
    digest_path = os.path.join(audit_dir, "manifest.sha256")
    if os.path.isfile(digest_path):
        recorded = open(digest_path, encoding="ascii").read().strip()
        actual = hashlib.sha256(open(manifest_path, "rb").read()).hexdigest()
        if recorded != actual:
            raise ValueError(
                f"manifest hash mismatch: file={recorded} vs recompute={actual}"
            )
    counts = manifest.get("counts", {})
    needed = {"train", "val", "test"}
    if not needed.issubset(counts):
        return {"ok": False, "pending_data": True,
                "reason": f"split thiếu: cần {needed}, có {set(counts)}"}
    if any(v == 0 for v in counts.values()):
        return {"ok": False, "pending_data": True,
                "reason": f"split có count=0: {counts}"}
    if manifest.get("holdout_status") != "human_curated_group_disjoint":
        return {"ok": False, "pending_data": True,
                "reason": f"holdout_status chưa phải human_curated: {manifest.get('holdout_status')}"}
    return {"ok": True, "manifest": manifest}


print("preflight helpers loaded")


In [ ]:
import os
from pathlib import Path
# Mặc định: dataset riêng cho mỗi bài toán. Trên Kaggle user attach cả 2 dataset.
HELMET_SLUG = os.environ.get("HELMET_DATASET_SLUG", "helmet-detect-v1")
EV_SLUG = os.environ.get("EV_DATASET_SLUG", "electric-vehicle-v1")
helmet_dir = os.path.join(KAGGLE_INPUT, HELMET_SLUG)
ev_dir = os.path.join(KAGGLE_INPUT, EV_SLUG)

# Preflight helmet
pf_helmet = preflight(helmet_dir)
if not pf_helmet["ok"]:
    raise SystemExit(f"helmet pending_data: {pf_helmet.get('reason')}")
print("helmet manifest ok:", pf_helmet["manifest"]["counts"])

# Preflight electric vehicle (nếu có slug)
if Path(ev_dir).is_dir():
    pf_ev = preflight(ev_dir)
    if not pf_ev["ok"]:
        raise SystemExit(f"electric-vehicle pending_data: {pf_ev.get('reason')}")
    print("ev manifest ok:", pf_ev["manifest"]["counts"])
else:
    print(f"skip ev: dataset {ev_dir} không có (R10 sẽ thu sau)")


In [ ]:
# Cell 1: train helmet detector
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
save_dir = WORKDIR / "runs" / "helmet_v1"
save_dir.mkdir(parents=True, exist_ok=True)
results = model.train(
    data=os.path.join(helmet_dir, "data.yaml"),
    epochs=int(os.environ.get("EPOCHS", "60")),
    imgsz=640,
    batch=16,
    cache=False,
    plots=False,
    project=str(WORKDIR / "runs"),
    name="helmet_v1",
    exist_ok=True,
    resume=False,
    device=0 if torch.cuda.is_available() else "cpu",
    verbose=False,
    seed=42,
)
print("helmet train done")


In [ ]:
# Cell 2: train electric-vehicle detector (chạy sau khi có dataset)
from ultralytics import YOLO
if not Path(ev_dir).is_dir():
    print("skip: không có EV slug trên Kaggle input")
else:
    model = YOLO("yolo11n.pt")
    results = model.train(
        data=os.path.join(ev_dir, "data.yaml"),
        epochs=int(os.environ.get("EPOCHS", "60")),
        imgsz=640,
        batch=16,
        cache=False,
        plots=False,
        project=str(WORKDIR / "runs"),
        name="electric_v1",
        exist_ok=True,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
        seed=42,
    )
    print("ev train done")


In [ ]:
# Export cả 2 model ONNX + manifest
import hashlib
import json
import shutil
from importlib import metadata
def _export_one(task: str, run_dir: Path, slug: str) -> dict | None:
    best = run_dir / "weights" / "best.pt"
    if not best.is_file():
        print(f"[{task}] no best.pt — skip export")
        return None
    onnx_path = WORKDIR / f"{slug}.onnx"
    from ultralytics import YOLO
    YOLO(str(best)).export(format="onnx", imgsz=640, opset=12, simplify=True)
    exported = Path(str(best).replace(".pt", ".onnx"))
    shutil.copy(exported, onnx_path)
    return {
        "task": task,
        "engine": "ultralytics.yolo11",
        "package_versions": {
            "ultralytics": metadata.version("ultralytics"),
            "torch": torch.__version__,
        },
        "onnx": str(onnx_path),
        "best_pt": str(best),
    }

manifests = []
for task, slug, run in [
    ("helmet", "helmet_v1", WORKDIR / "runs" / "helmet_v1"),
    ("electric_vehicle", "electric_v1", WORKDIR / "runs" / "electric_v1"),
]:
    out = _export_one(task, run, slug)
    if out:
        out["artifact_sha256"] = hashlib.sha256(open(out["onnx"], "rb").read()).hexdigest()
        out_path = WORKDIR / f"artifact_{task}.json"
        out_path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
        (WORKDIR / f"artifact_{task}.sha256").write_text(out["artifact_sha256"], encoding="ascii")
        manifests.append(out)
print("helmet/ev export done:", manifests)


In [ ]:
# Smoke test helmet (nếu có best.pt)
from pathlib import Path
helmet_dir_imgs = sorted(Path(helmet_dir) / "images" / "test").__iter__()
try:
    first = next(helmet_dir_imgs)
except StopIteration:
    print("no helmet test images; skip smoke")
else:
    onnx = WORKDIR / "helmet_v1.onnx"
    if not onnx.is_file():
        print("no helmet_v1.onnx; skip smoke")
    else:
        from ultralytics import YOLO
        res = YOLO(str(onnx)).predict(source=str(first), imgsz=640, conf=0.25, verbose=False)
        n_boxes = len(res[0].boxes) if res else 0
        print(f"helmet smoke: {first.name} -> {n_boxes} boxes")
